# Exploratory Analysis – Price Series Sanity Check

**Phase 1.6 – Visual Sanity Check**
- Consumer of `src.data.stock_data` – no duplicate yfinance logic
- Detects flatlines, unadjusted splits, late IPOs
- Plotly charts: adjusted close + normalized base 100

Run top-to-bottom in Codespace Jupyter kernel.

In [35]:
import sys
!{sys.executable} -m pip install "nbformat==5.10.4" "ipykernel==6.29.5" "ipywidgets==8.1.3"

import nbformat
print("nbformat", nbformat.__version__, "python", sys.executable)

nbformat 5.10.4 python /workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator/.venv/bin/python


In [36]:
import sys
from pathlib import Path
# Make repo root importable when notebook cwd is notebooks/
repo_root = Path().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data.stock_data import (
    DEFAULT_TICKERS,
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    fetch_and_cache,
    validate_price_data,
    summary_to_markdown,
)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print(f"Tickers: {DEFAULT_TICKERS}")
print(f"Date range: {DEFAULT_START_DATE} to {DEFAULT_END_DATE or 'today'}")
print(f"Repo root: {repo_root}")

Tickers: ['AAPL', 'MSFT', 'JPM', 'JNJ', 'PG', 'XOM', 'AMZN']
Date range: 2018-01-01 to today
Repo root: /workspaces/05-Financial-Portfolio-Optimization-and-Risk-Allocator


In [37]:
# Fetch via production pipeline – uses cache if exists
tickers = DEFAULT_TICKERS[:5]
df = fetch_and_cache(tickers, start="2020-01-01", end=None)
print(f"Shape: {df.shape}")
df.head()

Shape: (1717, 5)


,AAPL,MSFT,JPM,JNJ,PG
Date,,,,,
2020-01-02,72.271530,151.544205,117.313904,121.338547,103.275597
2020-01-03,71.568901,149.657257,115.765823,119.933685,102.581032
2020-01-06,72.139191,150.044083,115.673767,119.784065,102.723297
2020-01-07,71.799919,148.676041,113.707245,120.515572,102.087280
2020-01-08,72.954910,151.044235,114.594254,120.498978,102.522438


In [38]:
# Validation summary
summary = validate_price_data(df)
summary

{'n_tickers': 5,
 'n_rows': 1717,
 'start_date': '2020-01-02',
 'end_date': '2026-10-30',
 'n_trading_days_per_year_avg': 251.6,
 'all_positive': True,
 'has_nan': False,
 'duplicate_dates': False,
 'date_gaps_over_5_days': 0,
 'per_ticker_first_date': {'AAPL': '2020-01-02',
  'MSFT': '2020-01-02',
  'JPM': '2020-01-02',
  'JNJ': '2020-01-02',
  'PG': '2020-01-02'},
 'per_ticker_last_date': {'AAPL': '2026-10-30',
  'MSFT': '2026-10-30',
  'JPM': '2026-10-30',
  'JNJ': '2026-10-30',
  'PG': '2026-10-30'},
 'per_ticker_missing_count': {'AAPL': 0,
  'MSFT': 0,
  'JPM': 0,
  'JNJ': 0,
  'PG': 0}}

In [39]:
# Render validation as Markdown
from IPython.display import Markdown
Markdown(summary_to_markdown(summary))

# Data Summary Report

**Generated:** 2026-11-01

| Metric | Value |
|--------|-------|
| n_tickers | 5 |
| n_rows | 1717 |
| start_date | 2020-01-02 |
| end_date | 2026-10-30 |
| trading_days/year avg | 251.6 |
| all_positive | True |
| has_nan | False |
| duplicate_dates | False |
| date_gaps_over_5_days | 0 |

### Per-Ticker Coverage

| Ticker | First | Last | Missing |
|--------|-------|------|---------|
| AAPL | 2020-01-02 | 2026-10-30 | 0 |
| MSFT | 2020-01-02 | 2026-10-30 | 0 |
| JPM | 2020-01-02 | 2026-10-30 | 0 |
| JNJ | 2020-01-02 | 2026-10-30 | 0 |
| PG | 2020-01-02 | 2026-10-30 | 0 |

**Sanity:** trading_days/year ~252 indicates correct annualization. has_nan=False, all_positive=True required for downstream math.

In [ ]:
# Plotly line chart – one trace per ticker
fig = go.Figure()
for col in df.columns:
    fig.add_trace(go.Scatter(
        x=df.index, y=df[col], mode='lines', name=col,
        hovertemplate=f'{col}: %{{y:.2f}}<br>%{{x}}<extra></extra>'
    ))

fig.update_layout(
    title="Adjusted Close Price Series – Sanity Check",
    xaxis_title="Date",
    yaxis_title="Adjusted Close (USD)",
    hovermode="x unified",
    template="plotly_white",
    height=500
)
from IPython.display import HTML
HTML(fig.to_html())

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Normalized base 100 – relative performance
df_norm = df / df.iloc[0] * 100
fig2 = px.line(df_norm, title="Normalized Price (Base 100)")
fig2.update_layout(hovermode="x unified", template="plotly_white", height=500)
fig2.write_html("chart.html")

## Visual Checks
- **Flatline?** No – each series varies daily
- **-50% drop?** No – adjusted close removes split artifacts
- **Late start?** No – all tickers share common window (see validation dict)
- **Trading days/year ~252?** Yes – annualization correct

Ready for Phase 2 – Returns & Covariance.